# BAMO — صدای طبیعی فارسی + فیلم در Colab

**موتور صدا:** [Mana-Persian-Piper](https://huggingface.co/MahtaFetrat/Mana-Persian-Piper)  
آموزش‌دیده روی دیتاست Mana-TTS — طبیعی‌تر از Edge/MMS رباتیک

Runtime → Change runtime type → **GPU**

In [ ]:
# 1) نصب
!pip -q install piper-tts edge-tts soundfile numpy
!apt-get -qq install -y ffmpeg > /dev/null
!mkdir -p /content/mana_piper /content/bamo_out
!wget -q https://huggingface.co/MahtaFetrat/Mana-Persian-Piper/resolve/main/fa_IR-mana-medium.onnx -O /content/mana_piper/fa_IR-mana-medium.onnx
!wget -q https://huggingface.co/MahtaFetrat/Mana-Persian-Piper/resolve/main/fa_IR-mana-medium.onnx.json -O /content/mana_piper/fa_IR-mana-medium.onnx.json
print('ok')

In [ ]:
# 2) متن شورت (قابل ویرایش)
BEATS = [
    'صبر کن. این یکی عجیبه.',
    'صفر ممیز نه نه نه، تا بی نهایت.',
    'خیلی ها فکر می کنند از یک کمتر است.',
    'اما برابر یک است.',
    'فرض کن ایکس بشود صفر ممیز نه تکراری.',
    'ده برابرش می شود نه ممیز نه تکراری. بعد کم کن.',
    'نه ایکس می شود نه. پس ایکس می شود یک.',
    'ریاضی گاهی خلاف حس است، اما درست است.',
    'بامو. یوزر نیم ما دوازده بامو.',
]
CAPTIONS = [
    'صبر کن… عجیبه',
    '۰٫۹۹۹… تا بی‌نهایت',
    'فکر می‌کنی از ۱ کمتره؟',
    'اما = ۱',
    'x = ۰٫۹۹۹…',
    '۱۰x − x',
    '۹x = ۹ → x = ۱',
    'خلاف حس، اما درست',
    'بامو · @12bamo',
]

In [ ]:
# 3) تولید صدای طبیعی با Mana Piper
from pathlib import Path
import wave, json
import numpy as np
from piper import PiperVoice
from IPython.display import Audio, display

voice = PiperVoice.load('/content/mana_piper/fa_IR-mana-medium.onnx')
out_dir = Path('/content/bamo_out')
out_dir.mkdir(exist_ok=True)

def read_wav(path):
    with wave.open(str(path), 'rb') as w:
        sr = w.getframerate()
        audio = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16).astype(np.float32) / 32768.0
        return sr, audio

parts, timeline, t = [], [], 0.0
sr = None
for i, text in enumerate(BEATS):
    p = out_dir / f'p{i:02d}.wav'
    with wave.open(str(p), 'wb') as wf:
        voice.synthesize_wav(text, wf)
    srate, audio = read_wav(p)
    sr = srate
    dur = len(audio) / sr
    gap = 0.12 if i < len(BEATS)-1 else 0.25
    timeline.append({'i': i, 'start': round(t,3), 'speech_end': round(t+dur,3), 'end': round(t+dur+gap,3), 'caption': CAPTIONS[i]})
    parts.append(audio)
    parts.append(np.zeros(int(sr*gap), dtype=np.float32))
    t += dur + gap
    print(f'{i}: {dur:.2f}s  {text}')

full = np.concatenate(parts)
peak = np.max(np.abs(full)) + 1e-8
full = (full / peak * 0.95 * 32767).astype(np.int16)
narr = out_dir / 'narration.wav'
with wave.open(str(narr), 'wb') as w:
    w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
    w.writeframes(full.tobytes())
(out_dir / 'timeline.json').write_text(json.dumps(timeline, ensure_ascii=False, indent=2), encoding='utf-8')
print('TOTAL', round(t,2), 's')
display(Audio(str(narr)))

## 4) رندر فیلم
اگر ریپو BAMO را کلون کرده‌ای، از `ttv/render_sample.py` با این narration استفاده کن.
یا فقط فایل `bamo_out/narration.wav` را دانلود کن و روی ویدئو mux کن.

In [ ]:
# دانلود خروجی‌ها
from google.colab import files
files.download('/content/bamo_out/narration.wav')
files.download('/content/bamo_out/timeline.json')